In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("adm_pat_diag.csv")

# Extracting the features of procedures table and merging with the dataframe

In [3]:
# loading the procedures table
procedures = pd.read_csv("/Users/sujangauchan/Desktop/MIMIC csv/mimic-iv-3.1/hosp/procedures_icd.csv")

In [4]:
procedures

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
0,10000032,22595853,1,2180-05-07,5491,9
1,10000032,22841357,1,2180-06-27,5491,9
2,10000032,25742920,1,2180-08-06,5491,9
3,10000068,25022803,1,2160-03-03,8938,9
4,10000117,27988844,1,2183-09-19,0QS734Z,10
...,...,...,...,...,...,...
859650,19999840,21033226,5,2164-09-16,0331,9
859651,19999840,26071774,1,2164-07-25,8891,9
859652,19999840,26071774,2,2164-07-25,8841,9
859653,19999987,23865745,1,2145-11-07,8841,9


In [5]:
print(f'total unique patients: {procedures["subject_id"].nunique()}')
print(f'total unique admissions: {procedures['hadm_id'].nunique()}')

total unique patients: 150711
total unique admissions: 287504


In [6]:
# Get indices of min and max seq_num for each hadm_id
p_first_idx = procedures.groupby('hadm_id')['seq_num'].idxmin()
p_last_idx = procedures.groupby('hadm_id')['seq_num'].idxmax()

# Get first and last diagnoses values
p_result_1 = pd.DataFrame({
    'hadm_id': p_first_idx.index,
    'procedure_first_icd_code': procedures.loc[p_first_idx, 'icd_code'].values,
    'procedure_last_icd_code': procedures.loc[p_last_idx, 'icd_code'].values
})

In [7]:
# Get no of unique diagnoses counts, and the most frequent icd code
p_result_2 = procedures.groupby('hadm_id')['icd_code'].agg([
    ('procedures_total_count', 'count'),
    ('procedures_most_frequent_icd', lambda x: x.value_counts().index[0])
])

In [8]:
last_5_procedures = (procedures.sort_values(['hadm_id', 'seq_num'])
                   .groupby('hadm_id')
                   .tail(5)
                   .sort_values(['hadm_id', 'seq_num'], ascending=[True, False])  # Reverse seq_num order
                   .groupby('hadm_id')['icd_code']
                   .apply(lambda x: ' '.join(x.astype(str)))
                   .reset_index()
                   .rename(columns={'icd_code': 'procedures_last_5_icd_codes'}))

In [9]:
last_5_procedures 

,hadm_id,procedures_last_5_icd_codes
0,20000041,8154
1,20000045,3E0436Z
2,20000069,10E0XZZ 0KQM0ZZ
3,20000102,7309 7359
4,20000147,5A1221Z 06BQ4ZZ 021209W B211YZZ 02100Z9
...,...,...
287499,29999620,02H633Z 0JBR0ZZ 0JBR0ZZ
287500,29999625,02HV33Z 0BJ08ZZ 3E0G76Z 0DH63UZ 5A1955Z
287501,29999670,8856 0041 0046 3606 0066
287502,29999693,8E0W4CZ 0DB64Z3


In [12]:
# Get last 5 ICD codes as a single concatenated string
def pad_to_5_with_last(group):
    # Get all codes, reverse order (most recent first)
    codes = group['icd_code'].tolist()[::-1]
    # Pad with last code if needed
    while len(codes) < 5:
        codes.append(codes[0])
    return ' '.join(str(code) for code in codes[:5])

last_5_procedures_nempty = (procedures.sort_values(['hadm_id', 'seq_num'])
                    .groupby('hadm_id')
                    .apply(pad_to_5_with_last)
                    .reset_index()
                    .rename(columns={0: 'procedures_last_5_icd_codes'}))

last_5_procedures_nempty

/var/folders/kx/dxkg_2js1jj24wlz2dl_1rxh0000gn/T/ipykernel_62105/765245425.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pad_to_5_with_last)


,hadm_id,procedures_last_5_icd_codes
0,20000041,8154 8154 8154 8154 8154
1,20000045,3E0436Z 3E0436Z 3E0436Z 3E0436Z 3E0436Z
2,20000069,10E0XZZ 0KQM0ZZ 10E0XZZ 10E0XZZ 10E0XZZ
3,20000102,7309 7359 7309 7309 7309
4,20000147,5A1221Z 06BQ4ZZ 021209W B211YZZ 02100Z9
...,...,...
287499,29999620,02H633Z 0JBR0ZZ 0JBR0ZZ 02H633Z 02H633Z
287500,29999625,02HV33Z 0BJ08ZZ 3E0G76Z 0DH63UZ 5A1955Z
287501,29999670,8856 0041 0046 3606 0066
287502,29999693,8E0W4CZ 0DB64Z3 8E0W4CZ 8E0W4CZ 8E0W4CZ


In [10]:
procedures[procedures["hadm_id"]==29999620]

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
527760,16135001,29999620,1,2159-09-20,0JBR0ZZ,10
527761,16135001,29999620,2,2159-09-23,0JBR0ZZ,10
527762,16135001,29999620,3,2159-09-21,02H633Z,10


In [13]:
procedures_admission = p_result_1.merge(p_result_2, on = "hadm_id", how ="left").merge(last_5_procedures, on = "hadm_id", how = "left").merge(last_5_procedures_nempty, on = "hadm_id", how = "left")

In [14]:
procedures_admission

,hadm_id,procedure_first_icd_code,procedure_last_icd_code,procedures_total_count,procedures_most_frequent_icd,procedures_last_5_icd_codes_x,procedures_last_5_icd_codes_y
0,20000041,8154,8154,1,8154,8154,8154 8154 8154 8154 8154
1,20000045,3E0436Z,3E0436Z,1,3E0436Z,3E0436Z,3E0436Z 3E0436Z 3E0436Z 3E0436Z 3E0436Z
2,20000069,0KQM0ZZ,10E0XZZ,2,0KQM0ZZ,10E0XZZ 0KQM0ZZ,10E0XZZ 0KQM0ZZ 10E0XZZ 10E0XZZ 10E0XZZ
3,20000102,7359,7309,2,7359,7309 7359,7309 7359 7309 7309 7309
4,20000147,02100Z9,5A1221Z,5,02100Z9,5A1221Z 06BQ4ZZ 021209W B211YZZ 02100Z9,5A1221Z 06BQ4ZZ 021209W B211YZZ 02100Z9
...,...,...,...,...,...,...,...
287499,29999620,0JBR0ZZ,02H633Z,3,0JBR0ZZ,02H633Z 0JBR0ZZ 0JBR0ZZ,02H633Z 0JBR0ZZ 0JBR0ZZ 02H633Z 02H633Z
287500,29999625,009630Z,02HV33Z,7,009630Z,02HV33Z 0BJ08ZZ 3E0G76Z 0DH63UZ 5A1955Z,02HV33Z 0BJ08ZZ 3E0G76Z 0DH63UZ 5A1955Z
287501,29999670,0066,8856,5,0066,8856 0041 0046 3606 0066,8856 0041 0046 3606 0066
287502,29999693,0DB64Z3,8E0W4CZ,2,0DB64Z3,8E0W4CZ 0DB64Z3,8E0W4CZ 0DB64Z3 8E0W4CZ 8E0W4CZ 8E0W4CZ


In [ ]:
#merge extracted procedure features with admissions
df = df.merge(procedures_admission, on = "hadm_id", how = "left")

In [16]:
df

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,diagnoses_total_count,diagnoses_most_frequent_icd,diagnoses_last_5_icd_codes_x,diagnoses_last_5_icd_codes_y,procedure_first_icd_code,procedure_last_icd_code,procedures_total_count,procedures_most_frequent_icd,procedures_last_5_icd_codes_x,procedures_last_5_icd_codes_y
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,8.0,5723,V1582 30981 29680 496 07070,V1582 30981 29680 496 07070,5491,5491,1.0,5491,5491,5491 5491 5491 5491 5491
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,8.0,07071,3051 V08 5715 496 2761,3051 V08 5715 496 2761,5491,5491,1.0,5491,5491,5491 5491 5491 5491 5491
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,13.0,45829,5715 29680 496 V462 V4986,5715 29680 496 V462 V4986,NaN,NaN,NaN,NaN,NaN,NaN
3,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,10.0,07054,78791 3051 V08 496 2761,78791 3051 V08 496 2761,5491,5491,1.0,5491,5491,5491 5491 5491 5491 5491
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,1.0,30500,30500,30500 30500 30500 30500 30500,8938,8938,1.0,8938,8938,8938 8938 8938 8938 8938
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525775,19999784,21364559,2124-03-23 00:00:00,2124-03-29 13:16:00,NaN,ELECTIVE,P6717A,PHYSICIAN REFERRAL,HOME,Medicaid,...,12.0,Z5111,Z8619 E876 D708 T451X5A D701,Z8619 E876 D708 T451X5A D701,3E04305,3E04305,1.0,3E04305,3E04305,3E04305 3E04305 3E04305 3E04305 3E04305
525776,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,...,22.0,T8131XA,I9581 Z1611 B9620 Z87891 Z9049,I9581 Z1611 B9620 Z87891 Z9049,0HR7X74,3E0436Z,5.0,0HR7X74,3E0436Z 02HV33Z 0HBHXZZ 0HBJXZZ 0HR7X74,3E0436Z 02HV33Z 0HBHXZZ 0HBJXZZ 0HR7X74
525777,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,...,19.0,T8141XA,R197 E60 B954 E876 F419,R197 E60 B954 E876 F419,0J980ZZ,05HY33Z,3.0,0J980ZZ,05HY33Z 0WPF0JZ 0J980ZZ,05HY33Z 0WPF0JZ 0J980ZZ 05HY33Z 05HY33Z
525778,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,...,7.0,43491,3051 2724 4019 43811 34590,3051 2724 4019 43811 34590,8891,8841,2.0,8891,8841 8891,8841 8891 8841 8841 8841


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525780 entries, 0 to 525779
Data columns (total 37 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   subject_id                     525780 non-null  int64  
 1   hadm_id                        525780 non-null  int64  
 2   admittime                      525780 non-null  object 
 3   dischtime                      525780 non-null  object 
 4   deathtime                      0 non-null       float64
 5   admission_type                 525780 non-null  object 
 6   admit_provider_id              525776 non-null  object 
 7   admission_location             525779 non-null  object 
 8   discharge_location             376619 non-null  object 
 9   insurance                      516830 non-null  object 
 10  language                       525161 non-null  object 
 11  marital_status                 514189 non-null  object 
 12  race                          

In [18]:
# Lab events csv file size is 18 gb and cannot be loaded into ram of 8 gb only. Therefore, only loading 1 million rows for exploration
labevents1mill = pd.read_csv("/Users/sujangauchan/Desktop/MIMIC csv/mimic-iv-3.1/hosp/labevents.csv", 
                            nrows=100000)
labevents1mill

,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,1,10000032,NaN,2704548,50931,P69FQC,2180-03-23 11:51:00,2180-03-23 15:56:00,___,95.0,mg/dL,70.0,100.0,NaN,ROUTINE,"IF FASTING, 70-100 NORMAL, >125 PROVISIONAL DI..."
1,2,10000032,NaN,36092842,51071,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
2,3,10000032,NaN,36092842,51074,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
3,4,10000032,NaN,36092842,51075,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,BENZODIAZEPINE IMMUNOASSAY SCREEN DOES NOT DET...
4,5,10000032,NaN,36092842,51079,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,100407,10005866,20364112.0,42309100,50893,NaN,2149-10-04 01:59:00,2149-10-04 02:51:00,7.8,7.8,mg/dL,8.4,10.3,abnormal,ROUTINE,NaN
99996,100408,10005866,20364112.0,42309100,50902,NaN,2149-10-04 01:59:00,2149-10-04 02:51:00,100,100.0,mEq/L,96.0,108.0,NaN,ROUTINE,NaN
99997,100409,10005866,20364112.0,42309100,50912,NaN,2149-10-04 01:59:00,2149-10-04 02:51:00,0.4,0.4,mg/dL,0.5,1.2,abnormal,ROUTINE,NaN
99998,100410,10005866,20364112.0,42309100,50931,NaN,2149-10-04 01:59:00,2149-10-04 02:51:00,___,131.0,mg/dL,70.0,100.0,abnormal,ROUTINE,"If fasting, 70-100 normal, >125 provisional di..."


In [19]:
def count_normal_abnormal_by_hadm(file_path, chunk_size=100000):
    from collections import defaultdict
    
    normal_counts = defaultdict(int)
    abnormal_counts = defaultdict(int)
    chunk_count = 0
    total_normal = 0
    total_abnormal = 0
    
    print("Processing labevents to count normal and abnormal flags by hadm_id...")
    
    # Only read necessary columns to save memory
    columns_needed = ['hadm_id', 'flag']
    
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, usecols=columns_needed):
        chunk_count += 1
        
        # Remove rows with NaN hadm_id
        chunk_clean = chunk[chunk['hadm_id'].notna()].copy()
        
        if not chunk_clean.empty:
            # Count normal flags (blank/NaN values in flag column)
            normal_chunk = chunk_clean[chunk_clean['flag'].isna() | (chunk_clean['flag'] == '')]
            for hadm_id in normal_chunk['hadm_id']:
                normal_counts[int(hadm_id)] += 1
                total_normal += 1
            
            # Count abnormal flags
            abnormal_chunk = chunk_clean[chunk_clean['flag'] == 'abnormal']
            for hadm_id in abnormal_chunk['hadm_id']:
                abnormal_counts[int(hadm_id)] += 1
                total_abnormal += 1
        
        # Progress update
        if chunk_count % 200 == 0:
            print(f"Processed {chunk_count} chunks:")
            print(f"  - {total_normal:,} normal flags ({len(normal_counts)} unique hadm_ids)")
            print(f"  - {total_abnormal:,} abnormal flags ({len(abnormal_counts)} unique hadm_ids)")
    
    # Get all unique hadm_ids
    all_hadm_ids = set(normal_counts.keys()) | set(abnormal_counts.keys())
    
    # Create final DataFrame
    result_data = []
    for hadm_id in all_hadm_ids:
        result_data.append({
            'hadm_id': hadm_id,
            'Normal_counts': normal_counts.get(hadm_id, 0),
            'Abnormal_counts': abnormal_counts.get(hadm_id, 0)
        })
    
    result_df = pd.DataFrame(result_data)
    result_df = result_df.sort_values('hadm_id').reset_index(drop=True)
    
    # Add total column
    result_df['Total_lab_events'] = result_df['Normal_counts'] + result_df['Abnormal_counts']
    
    print(f"\nFinal results:")
    print(f"Total normal flags: {total_normal:,}")
    print(f"Total abnormal flags: {total_abnormal:,}")
    print(f"Unique hadm_ids: {len(result_df)}")
    
    return result_df

# Usage
lab_counts_df = count_normal_abnormal_by_hadm("/Users/sujangauchan/Desktop/MIMIC csv/mimic-iv-3.1/hosp/labevents.csv")

# Display results
lab_counts_df

Processing labevents to count normal and abnormal flags by hadm_id...
Processed 200 chunks:
  - 6,876,142 normal flags (56312 unique hadm_ids)
  - 3,841,322 abnormal flags (55322 unique hadm_ids)
Processed 400 chunks:
  - 13,737,058 normal flags (112190 unique hadm_ids)
  - 7,674,547 abnormal flags (110299 unique hadm_ids)
Processed 600 chunks:
  - 20,620,963 normal flags (168288 unique hadm_ids)
  - 11,513,041 abnormal flags (165478 unique hadm_ids)
Processed 800 chunks:
  - 27,453,256 normal flags (224471 unique hadm_ids)
  - 15,326,351 abnormal flags (220699 unique hadm_ids)
Processed 1000 chunks:
  - 34,308,622 normal flags (280438 unique hadm_ids)
  - 19,144,390 abnormal flags (275738 unique hadm_ids)
Processed 1200 chunks:
  - 41,129,215 normal flags (336844 unique hadm_ids)
  - 22,938,493 abnormal flags (331280 unique hadm_ids)
Processed 1400 chunks:
  - 47,987,000 normal flags (393347 unique hadm_ids)
  - 26,737,720 abnormal flags (386776 unique hadm_ids)

Final results:
Total 

,hadm_id,Normal_counts,Abnormal_counts,Total_lab_events
0,20000019,52,23,75
1,20000024,22,6,28
2,20000034,76,54,130
3,20000041,48,18,66
4,20000045,151,133,284
...,...,...,...,...
447684,29999803,436,225,661
447685,29999809,176,94,270
447686,29999828,52,21,73
447687,29999928,32,26,58


In [20]:
lab_counts_df['Total_lab_events'].sum()

84605867

In [21]:
lab_counts_df['Normal_counts'].sum()

54316219

In [22]:
lab_counts_df['Abnormal_counts'].sum()

30289648

In [27]:
df = df.merge(lab_counts_df, on = "hadm_id", how= "left")

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525780 entries, 0 to 525779
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   subject_id                     525780 non-null  int64  
 1   hadm_id                        525780 non-null  int64  
 2   admittime                      525780 non-null  object 
 3   dischtime                      525780 non-null  object 
 4   deathtime                      0 non-null       float64
 5   admission_type                 525780 non-null  object 
 6   admit_provider_id              525776 non-null  object 
 7   admission_location             525779 non-null  object 
 8   discharge_location             376619 non-null  object 
 9   insurance                      516830 non-null  object 
 10  language                       525161 non-null  object 
 11  marital_status                 514189 non-null  object 
 12  race                          

In [29]:
df["abnormal%"] = df["Abnormal_counts"] / df["Total_lab_events"]

In [30]:
df.to_csv('proc_labevents.csv', index = False)

In [31]:
df

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,procedure_first_icd_code,procedure_last_icd_code,procedures_total_count,procedures_most_frequent_icd,procedures_last_5_icd_codes_x,procedures_last_5_icd_codes_y,Normal_counts,Abnormal_counts,Total_lab_events,abnormal%
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,5491,5491,1.0,5491,5491,5491 5491 5491 5491 5491,38.0,19.0,57.0,0.333333
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,5491,5491,1.0,5491,5491,5491 5491 5491 5491 5491,27.0,19.0,46.0,0.413043
2,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,NaN,NaN,NaN,NaN,NaN,NaN,29.0,39.0,68.0,0.573529
3,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,5491,5491,1.0,5491,5491,5491 5491 5491 5491 5491,29.0,40.0,69.0,0.579710
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,8938,8938,1.0,8938,8938,8938 8938 8938 8938 8938,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525775,19999784,21364559,2124-03-23 00:00:00,2124-03-29 13:16:00,NaN,ELECTIVE,P6717A,PHYSICIAN REFERRAL,HOME,Medicaid,...,3E04305,3E04305,1.0,3E04305,3E04305,3E04305 3E04305 3E04305 3E04305 3E04305,206.0,65.0,271.0,0.239852
525776,19999828,29734428,2147-07-18 16:23:00,2147-08-04 18:10:00,NaN,EW EMER.,P38XL8,PHYSICIAN REFERRAL,HOME HEALTH CARE,Medicaid,...,0HR7X74,3E0436Z,5.0,0HR7X74,3E0436Z 02HV33Z 0HBHXZZ 0HBJXZZ 0HR7X74,3E0436Z 02HV33Z 0HBHXZZ 0HBJXZZ 0HR7X74,355.0,178.0,533.0,0.333959
525777,19999828,25744818,2149-01-08 16:44:00,2149-01-18 17:00:00,NaN,EW EMER.,P13JMH,TRANSFER FROM HOSPITAL,HOME HEALTH CARE,Medicaid,...,0J980ZZ,05HY33Z,3.0,0J980ZZ,05HY33Z 0WPF0JZ 0J980ZZ,05HY33Z 0WPF0JZ 0J980ZZ 05HY33Z 05HY33Z,227.0,139.0,366.0,0.379781
525778,19999840,26071774,2164-07-25 00:27:00,2164-07-28 12:15:00,NaN,EW EMER.,P036NA,EMERGENCY ROOM,HOME,Private,...,8891,8841,2.0,8891,8841 8891,8841 8891 8841 8841 8841,126.0,28.0,154.0,0.181818
